# calibration: camera intrinsics from a ChArUco board (M1)

A ChArUco board is a chessboard with ArUco markers in the white squares. The markers make
every inner chessboard corner identifiable even when the board is partly out of view, so
each photo gives a set of (3D board point, 2D image point) pairs. `cv2.calibrateCamera`
then fits the camera matrix K (fx, fy, cx, cy) and the lens distortion coefficients by
minimizing the reprojection error over all views.

Output: `calibration/camera_intrinsics.yaml` (path from `configs/camera.yaml:
intrinsics_file`), used to undistort frames before marker detection (`homography.ipynb`).
The board itself is printed with `markers.ipynb`; photos are taken in `run_calibrate.ipynb`.

Milestone: M1. Deliverable: intrinsics + reprojection error report.

**TODO**
- [ ] `collect_views`: live capture loop (show detections, keep a frame on keypress, reject
      blurry views and views with < min_corners corners)
- [ ] `coverage_report`: check the views cover the whole image (corners and edges) and a
      range of tilts; poor coverage -> unreliable distortion estimate
- [ ] decide on the distortion model (default 5 coefficients; try `CALIB_RATIONAL_MODEL`
      only if the error stays high) and an acceptance threshold (e.g. RMS < 0.5 px)

In [ ]:
from __future__ import annotations

import datetime as _dt
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import yaml

%run ./config.ipynb

## The ChArUco board
Geometry from `configs/camera.yaml: charuco` (squares, square/marker length in meters,
dictionary). Measure the printed square length and update the config, because the scale
of the 3D points comes from it (it affects the extrinsics, not K).

In [ ]:
def make_charuco_board(charuco_cfg=None):
    """cv2.aruco.CharucoBoard from the charuco section of camera.yaml."""
    import cv2
    c = charuco_cfg or load_config("configs/camera.yaml").charuco
    dictionary = cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, c.dictionary))
    return cv2.aruco.CharucoBoard((int(c.squares_x), int(c.squares_y)), float(c.square_length),
                                  float(c.marker_length), dictionary)


def detect_charuco(image: np.ndarray, board, min_corners: int = 4):
    """Detect ChArUco corners in one image.

    Returns (corners (N, 1, 2) float32, ids (N, 1) int32), or (None, None) if fewer than
    min_corners were found.
    """
    import cv2
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    detector = cv2.aruco.CharucoDetector(board)
    corners, ids, _marker_corners, _marker_ids = detector.detectBoard(gray)
    if ids is None or len(ids) < min_corners:
        return None, None
    return corners, ids

## Calibration and reprojection error
The reprojection error of a view is the RMS pixel distance between the detected corners and
the board points projected with the fitted K, distortion and that view's pose. Report the
overall RMS and look for outlier views (bad detections, motion blur) to drop.

In [ ]:
@dataclass
class CalibrationResult:
    rms: float                    # overall RMS reprojection error (px), from calibrateCamera
    camera_matrix: np.ndarray     # (3, 3)
    dist_coeffs: np.ndarray       # (k,)
    image_size: tuple[int, int]   # (width, height)
    per_view_errors: list[float] = field(default_factory=list)
    rvecs: list = field(default_factory=list, repr=False)
    tvecs: list = field(default_factory=list, repr=False)


def reprojection_errors(obj_points, img_points, rvecs, tvecs, K, dist) -> list[float]:
    """Per-view RMS reprojection error in pixels."""
    import cv2
    errs = []
    for obj, img, r, t in zip(obj_points, img_points, rvecs, tvecs):
        proj, _ = cv2.projectPoints(np.asarray(obj, np.float32), r, t, K, dist)
        d = proj.reshape(-1, 2) - np.asarray(img, float).reshape(-1, 2)
        errs.append(float(np.sqrt((d ** 2).sum(axis=1).mean())))
    return errs


def calibrate_from_points(obj_points, img_points, image_size: tuple[int, int],
                          flags: int = 0) -> CalibrationResult:
    """cv2.calibrateCamera wrapper. obj_points: list of (N, 3); img_points: list of (N, 2)."""
    import cv2
    obj = [np.asarray(o, np.float32).reshape(-1, 1, 3) for o in obj_points]
    img = [np.asarray(i, np.float32).reshape(-1, 1, 2) for i in img_points]
    if len(obj) < 3:
        raise ValueError(f"need at least 3 views, got {len(obj)}")
    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(obj, img, tuple(image_size), None, None,
                                                     flags=flags)
    return CalibrationResult(float(rms), K, dist.ravel(), tuple(int(v) for v in image_size),
                             reprojection_errors(obj, img, rvecs, tvecs, K, dist),
                             list(rvecs), list(tvecs))


def calibrate_charuco(views, board, image_size: tuple[int, int], flags: int = 0) -> CalibrationResult:
    """views: list of (charuco_corners, charuco_ids) from detect_charuco (None entries skipped)."""
    obj_points, img_points = [], []
    for corners, ids in views:
        if corners is None:
            continue
        obj, img = board.matchImagePoints(corners, ids)
        if obj is None or len(obj) < 6:   # calibrateCamera needs >= 4 non-collinear points
            continue
        obj_points.append(obj.reshape(-1, 3))
        img_points.append(img.reshape(-1, 2))
    return calibrate_from_points(obj_points, img_points, image_size, flags)

## Saving / loading the intrinsics
Plain YAML (lists, not OpenCV FileStorage) so any notebook and the ROS side can read it.

In [ ]:
def default_intrinsics_path() -> Path:
    return gv_path(load_config("configs/camera.yaml").intrinsics_file)


def save_intrinsics(result: CalibrationResult, path: str | Path | None = None) -> Path:
    path = Path(path) if path is not None else default_intrinsics_path()
    path.parent.mkdir(parents=True, exist_ok=True)
    data = {
        "image_size": list(result.image_size),
        "camera_matrix": np.asarray(result.camera_matrix, float).tolist(),
        "dist_coeffs": np.asarray(result.dist_coeffs, float).ravel().tolist(),
        "rms_reprojection_error_px": float(result.rms),
        "per_view_errors_px": [float(e) for e in result.per_view_errors],
        "n_views": len(result.per_view_errors),
        "date": _dt.date.today().isoformat(),
    }
    path.write_text(yaml.safe_dump(data, sort_keys=False))
    return path


def load_intrinsics(path: str | Path | None = None) -> dict:
    """Returns {"K": (3, 3), "dist": (k,), "image_size": (w, h), "rms": float}."""
    path = Path(path) if path is not None else default_intrinsics_path()
    d = yaml.safe_load(Path(path).read_text())
    return {"K": np.array(d["camera_matrix"], float), "dist": np.array(d["dist_coeffs"], float),
            "image_size": tuple(d["image_size"]), "rms": d.get("rms_reprojection_error_px")}


def undistort(image: np.ndarray, K: np.ndarray, dist: np.ndarray) -> np.ndarray:
    """Undistorted image with the same K (fine for a narrow-ish lens).

    TODO M1: for speed in the live loop, precompute maps once with
    cv2.initUndistortRectifyMap and use cv2.remap per frame.
    """
    import cv2
    return cv2.undistort(image, K, dist)

## Collection workflow (TODO)

In [ ]:
def collect_views(camera, board, n_views: int = 25, min_corners: int = 12):
    """TODO M1: interactive capture of ChArUco views.

    camera: an opened cv2.VideoCapture. Loop: read frame, detect_charuco, draw the detected
    corners (cv2.aruco.drawDetectedCornersCharuco), show it; on a key press keep the frame if it
    has >= min_corners corners and is sharp (variance of the Laplacian above a threshold).
    Stop after n_views. Return (frames, views) where views = [(corners, ids), ...].
    Hints: move the board to all image corners and tilt it +-30..45 degrees; keep it still.
    """
    raise NotImplementedError("TODO M1: collect_views")


def coverage_report(views, image_size: tuple[int, int], grid: int = 4) -> dict:
    """TODO M1: fraction of a grid x grid tiling of the image that contains detected corners,
    plus the list of empty tiles, so the student knows where to hold the board next."""
    raise NotImplementedError("TODO M1: coverage_report")

## Self-check
1. Detect corners on a rendered ChArUco board image.
2. Calibrate from synthetic projections of the board with a known K and check K comes back.
3. YAML round trip in a temporary directory.

In [ ]:
import tempfile

import cv2

_cal_board = make_charuco_board()
_cal_cfg = load_config("configs/camera.yaml").charuco
_n_inner = (_cal_cfg.squares_x - 1) * (_cal_cfg.squares_y - 1)

# 1. detection on a clean render (with a white margin)
_img = cv2.copyMakeBorder(_cal_board.generateImage((700, 500)), 40, 40, 40, 40,
                          cv2.BORDER_CONSTANT, value=255)
_corners, _ids = detect_charuco(_img, _cal_board)
assert _ids is not None and len(_ids) == _n_inner, (None if _ids is None else len(_ids))
_obj, _imgp = _cal_board.matchImagePoints(_corners, _ids)
assert _obj.reshape(-1, 3).shape == (_n_inner, 3)

# 2. synthetic calibration with known intrinsics
_K_true = np.array([[600.0, 0, 320], [0, 610.0, 240], [0, 0, 1]])
_pts3d = _cal_board.getChessboardCorners().reshape(-1, 3)
_rng = np.random.default_rng(0)
_objs, _imgs = [], []
for _i in range(10):
    _rvec = np.array([_rng.uniform(-0.5, 0.5), _rng.uniform(-0.5, 0.5), _rng.uniform(-0.3, 0.3)])
    _tvec = np.array([_rng.uniform(-0.12, 0.0), _rng.uniform(-0.08, 0.0), _rng.uniform(0.35, 0.5)])
    _p, _ = cv2.projectPoints(_pts3d, _rvec, _tvec, _K_true, np.zeros(5))
    _objs.append(_pts3d)
    _imgs.append(_p.reshape(-1, 2) + _rng.normal(0, 0.1, (len(_pts3d), 2)))
_res = calibrate_from_points(_objs, _imgs, (640, 480))
assert abs(_res.camera_matrix[0, 0] - 600) / 600 < 0.02, _res.camera_matrix
assert abs(_res.camera_matrix[1, 1] - 610) / 610 < 0.02
assert _res.rms < 0.5 and len(_res.per_view_errors) == 10 and max(_res.per_view_errors) < 0.5

# 3. YAML round trip
with tempfile.TemporaryDirectory() as _t:
    _path = save_intrinsics(_res, Path(_t) / "calibration" / "camera_intrinsics.yaml")
    _loaded = load_intrinsics(_path)
    assert np.allclose(_loaded["K"], _res.camera_matrix) and _loaded["image_size"] == (640, 480)
    assert np.allclose(_loaded["dist"], _res.dist_coeffs)
assert undistort(_img, _loaded["K"], _loaded["dist"]).shape == _img.shape
print(f"calibration OK: synthetic fx={_res.camera_matrix[0, 0]:.1f} fy={_res.camera_matrix[1, 1]:.1f} rms={_res.rms:.3f}px")